# 🤖 Local Agentic RAG with Ollama, ChromaDB, and LangGraph

**A fully local, privacy-preserving Retrieval-Augmented Generation pipeline**

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)]()

---

## What You'll Learn

In this notebook, you will build a **minimal end-to-end Agentic RAG pipeline** that runs entirely on your local machine — no internet connection or cloud API required after setup.

### 🔍 What is RAG?

**Retrieval-Augmented Generation (RAG)** is a technique that enhances a language model's responses by first *retrieving* relevant documents from a knowledge base, then *augmenting* the prompt with that context before generating an answer. This is especially useful when:

- Your LLM doesn't know about domain-specific or up-to-date information
- You want answers grounded in specific documents (e.g., internal manuals, research papers)
- You want to reduce hallucinations

### 🤖 What Makes It "Agentic"?

A standard RAG pipeline is a fixed sequence: retrieve → generate. An **Agentic RAG** pipeline uses a reasoning loop where the model can:

- Decide *whether* retrieval is needed
- Call tools (e.g., a calculator, search function)
- Reflect on intermediate results before producing a final answer

We implement this loop using **LangGraph**, a lightweight graph-based orchestration library.

### 🔒 Why Local-First?

Running AI entirely on your own hardware provides:

- **Privacy** — your data never leaves your machine
- **Offline capability** — works without internet
- **Cost control** — no per-token API charges
- **Reproducibility** — same model version every run

### ⚡ Where Does OpenVINO Fit?

[OpenVINO™](https://github.com/openvinotoolkit/openvino) is Intel's open-source toolkit for optimizing and deploying deep learning models. It can accelerate LLM inference on Intel CPUs, iGPUs, and NPUs by:

- Quantizing models (e.g., INT4/INT8) to reduce memory usage
- Compiling computation graphs for hardware-specific optimization
- Enabling faster token generation on CPU compared to vanilla PyTorch

In this notebook, OpenVINO is shown as an **optional enhancement** — the pipeline runs fine without it, and we show how to plug it in.

---

### 🗺️ Pipeline Overview

```
User Query
    │
    ▼
┌─────────────────────────────────────────┐
│           LangGraph Agent Loop          │
│                                         │
│  ┌──────────┐     ┌──────────────────┐  │
│  │  Decide  │───▶│  Retrieve Docs    │  │
│  │  (LLM)   │     │  (ChromaDB)      │  │
│  └──────────┘     └────────┬─────────┘  │
│       ▲                    │            │
│       │            ┌───────▼──────────┐ │
│       └────────────│  Generate Answer │ │
│                    │  (Ollama LLM)    │ │
│                    └──────────────────┘ │
└─────────────────────────────────────────┘
    │
    ▼
Final Answer
```

---

### 📋 Prerequisites

| Requirement | Minimum | Recommended |
|---|---|---|
| RAM | 8 GB | 16 GB |
| CPU | Any x86-64 / ARM64 | Intel Core i5+ |
| Storage | 5 GB free | 10 GB free |
| OS | Linux / macOS / Windows | Ubuntu 22.04+ |
| Python | 3.9+ | 3.11 |

> ✅ **No GPU required.** This notebook is designed to run on CPU only.

# 🤖 Local Agentic RAG with Ollama, ChromaDB, and LangGraph

**A fully local, privacy-preserving Retrieval-Augmented Generation pipeline**

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)]()

---

## What You'll Learn

In this notebook, you will build a **minimal end-to-end Agentic RAG pipeline** that runs entirely on your local machine — no internet connection or cloud API required after setup.

### 🔍 What is RAG?

**Retrieval-Augmented Generation (RAG)** is a technique that enhances a language model's responses by first *retrieving* relevant documents from a knowledge base, then *augmenting* the prompt with that context before generating an answer. This is especially useful when:

- Your LLM doesn't know about domain-specific or up-to-date information
- You want answers grounded in specific documents (e.g., internal manuals, research papers)
- You want to reduce hallucinations

### 🤖 What Makes It "Agentic"?

A standard RAG pipeline is a fixed sequence: retrieve → generate. An **Agentic RAG** pipeline uses a reasoning loop where the model can:

- Decide *whether* retrieval is needed
- Call tools (e.g., a calculator, search function)
- Reflect on intermediate results before producing a final answer

We implement this loop using **LangGraph**, a lightweight graph-based orchestration library.

### 🔒 Why Local-First?

Running AI entirely on your own hardware provides:

- **Privacy** — your data never leaves your machine
- **Offline capability** — works without internet
- **Cost control** — no per-token API charges
- **Reproducibility** — same model version every run

### ⚡ Where Does OpenVINO Fit?

[OpenVINO™](https://github.com/openvinotoolkit/openvino) is Intel's open-source toolkit for optimizing and deploying deep learning models. It can accelerate LLM inference on Intel CPUs, iGPUs, and NPUs by:

- Quantizing models (e.g., INT4/INT8) to reduce memory usage
- Compiling computation graphs for hardware-specific optimization
- Enabling faster token generation on CPU compared to vanilla PyTorch

In this notebook, OpenVINO is shown as an **optional enhancement** — the pipeline runs fine without it, and we show how to plug it in.

---

### 🗺️ Pipeline Overview

```
User Query
    │
    ▼
┌─────────────────────────────────────────┐
│           LangGraph Agent Loop          │
│                                         │
│  ┌──────────┐     ┌──────────────────┐  │
│  │  Decide  │───▶│  Retrieve Docs    │  │
│  │  (LLM)   │     │  (ChromaDB)      │  │
│  └──────────┘     └────────┬─────────┘  │
│       ▲                    │            │
│       │            ┌───────▼──────────┐ │
│       └────────────│  Generate Answer │ │
│                    │  (Ollama LLM)    │ │
│                    └──────────────────┘ │
└─────────────────────────────────────────┘
    │
    ▼
Final Answer
```

---

### 📋 Prerequisites

| Requirement | Minimum | Recommended |
|---|---|---|
| RAM | 8 GB | 16 GB |
| CPU | Any x86-64 / ARM64 | Intel Core i5+ |
| Storage | 5 GB free | 10 GB free |
| OS | Linux / macOS / Windows | Ubuntu 22.04+ |
| Python | 3.9+ | 3.11 |

> ✅ **No GPU required.** This notebook is designed to run on CPU only.

---
## 📦 Section 1: Environment Setup

We install the required Python packages. These are all lightweight and widely used:

| Package | Purpose |
|---|---|
| `ollama` | Python client for local Ollama LLM server |
| `chromadb` | Local vector database for document embeddings |
| `langgraph` | Agent loop orchestration |
| `langchain-community` | Utility helpers (text splitters, etc.) |
| `sentence-transformers` | Lightweight local embedding model |
| `tqdm` | Progress bars |


In [1]:
# Install required packages
# This may take 1-2 minutes on first run
%pip install -q \
    ollama \
    chromadb \
    langgraph \
    langchain \
    langchain-community \
    sentence-transformers \
    tqdm

print("✅ All packages installed successfully.")

Note: you may need to restart the kernel to use updated packages.
✅ All packages installed successfully.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

### 🦙 Installing and Starting Ollama

**Ollama** is a tool for running large language models locally. It handles model download, quantization, and serves an OpenAI-compatible REST API on `http://localhost:11434`.

**Installation:**

```bash
# Linux / macOS:
curl -fsSL https://ollama.com/install.sh | sh

# Windows: Download installer from https://ollama.com/download
```

**Start the Ollama server** (in a separate terminal or as a background service):

```bash
ollama serve
```

**Pull a lightweight model** suitable for CPU inference:

```bash
# ~2.3 GB — good balance of quality and speed on CPU
ollama pull qwen2.5:3b

# Smaller alternative (~1.1 GB) if RAM is limited:
# ollama pull qwen2.5:1.5b
```

> 💡 **Why Qwen2.5?** It delivers strong instruction-following performance in small sizes (1.5B–7B), making it ideal for CPU-only environments. The 3B variant comfortably fits in 8 GB RAM.

After pulling the model, verify Ollama is running by executing the next cell.

In [2]:
import ollama

# ── Configuration ────────────────────────────────────────────────────────────
# Change MODEL_NAME to match whichever model you pulled with `ollama pull`
MODEL_NAME = "qwen2.5:3b"       # Recommended: good quality on CPU
# MODEL_NAME = "qwen2.5:1.5b"  # Uncomment for lower RAM usage (~1.1 GB)
# MODEL_NAME = "llama3.2:3b"    # Alternative: Meta LLaMA 3.2 3B
# ─────────────────────────────────────────────────────────────────────────────

# Verify Ollama is running and the model is available
try:
    available_models = [m.model for m in ollama.list().models]
    print("🦙 Ollama is running!")
    print(f"   Available models: {available_models}")
    
    if MODEL_NAME not in available_models:
        print(f"\n⚠️  Model '{MODEL_NAME}' not found locally.")
        print(f"   Please run in a terminal:  ollama pull {MODEL_NAME}")
    else:
        print(f"\n✅ Model '{MODEL_NAME}' is ready to use.")
        
except Exception as e:
    print(f"❌ Could not connect to Ollama: {e}")
    print("   Make sure Ollama is installed and running: `ollama serve`")

🦙 Ollama is running!
   Available models: ['qwen2.5:3b', 'gemma3:4b', 'mxbai-embed-large:latest', 'gemma3:latest']

✅ Model 'qwen2.5:3b' is ready to use.


---
## 💬 Section 2: Basic LLM Inference with Ollama

Before building the full pipeline, let's confirm the LLM works with a simple prompt-response test.

We use the `ollama` Python client, which communicates with the locally running Ollama server. The interface mirrors the OpenAI Chat Completions API, so the concepts transfer directly.

In [3]:
def ask_llm(prompt: str, model: str = MODEL_NAME, system: str = None) -> str:
    """
    Send a prompt to the local Ollama LLM and return the response text.
    
    Args:
        prompt:  The user message / question.
        model:   Ollama model name to use.
        system:  Optional system prompt to set the assistant's behaviour.
    
    Returns:
        The model's response as a plain string.
    """
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    
    response = ollama.chat(model=model, messages=messages)
    return response["message"]["content"]


# ── Simple test ───────────────────────────────────────────────────────────────
print("Sending a test prompt to the LLM...\n")

test_response = ask_llm(
    prompt="In one sentence, what is Intel OpenVINO?",
    system="You are a concise technical assistant."
)

print(f"LLM Response:\n{test_response}")

Sending a test prompt to the LLM...

LLM Response:
Intel OpenVINO is an open-source platform optimized for real-time inference of deep learning models on resource-constrained devices.


---
## 📄 Section 3: Document Preparation

For RAG to work, we need a **knowledge base** — a set of documents the system can search through to answer questions.

In this example, we use a small set of manually written paragraphs about Intel OpenVINO and related AI topics. In a real project, you would replace these with:

- PDF or text files loaded from disk
- Web pages fetched via scraping
- Database records
- Any structured or unstructured text

We also split long documents into smaller **chunks**. This is important because:

1. Embedding models have a maximum input length (typically 256–512 tokens)
2. Smaller chunks improve retrieval precision — you return only the relevant paragraph, not an entire page
3. The LLM context window is limited; smaller chunks fit more retrieved results

In [4]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# ── Sample knowledge base ─────────────────────────────────────────────────────
# These short documents form our local knowledge base.
# Replace or extend with your own content.

RAW_DOCUMENTS = [
    {
        "id": "doc_openvino_overview",
        "text": (
            "Intel OpenVINO (Open Visual Inference and Neural network Optimization) is an open-source "
            "toolkit for optimizing and deploying AI inference. It supports models from frameworks "
            "like PyTorch, TensorFlow, and ONNX. OpenVINO converts models into its Intermediate "
            "Representation (IR) format for cross-hardware deployment. It targets Intel CPUs, "
            "integrated GPUs, and NPUs. Key features include model quantization (INT8/INT4), "
            "throughput optimization, and a Python API for easy integration."
        ),
        "source": "openvino_overview",
    },
    {
        "id": "doc_rag_explanation",
        "text": (
            "Retrieval-Augmented Generation (RAG) is a technique that improves LLM responses by "
            "fetching relevant documents from an external knowledge base before generating an answer. "
            "The workflow has two stages: retrieval, where a query is converted to an embedding and "
            "matched against stored document embeddings in a vector database; and generation, where "
            "the retrieved documents are concatenated with the original query as context for the LLM. "
            "RAG reduces hallucinations and allows the model to answer questions about private or "
            "domain-specific data without fine-tuning."
        ),
        "source": "rag_explanation",
    },
    {
        "id": "doc_chromadb",
        "text": (
            "ChromaDB is a lightweight, open-source vector database designed for AI applications. "
            "It stores document embeddings (dense numerical vectors) and supports fast similarity "
            "search using cosine or L2 distance. ChromaDB can run fully in-memory for prototyping "
            "or persist data to disk for production use. It integrates natively with popular "
            "embedding models from HuggingFace and OpenAI, and requires no separate database server."
        ),
        "source": "chromadb_overview",
    },
    {
        "id": "doc_langgraph",
        "text": (
            "LangGraph is a library for building stateful, multi-step agent workflows using a "
            "graph-based computation model. Nodes represent individual actions (e.g., call LLM, "
            "retrieve documents, execute tool), and edges define the control flow between them. "
            "LangGraph supports conditional routing, loops, and human-in-the-loop checkpoints. "
            "It is part of the LangChain ecosystem and can be used with any LLM provider, "
            "including locally running models via Ollama."
        ),
        "source": "langgraph_overview",
    },
    {
        "id": "doc_ollama",
        "text": (
            "Ollama is a tool for running large language models locally on your machine. It "
            "provides a simple CLI and REST API for downloading and serving quantized models "
            "in GGUF format. Supported models include Llama 3, Qwen2.5, Mistral, Phi-3, and "
            "many others. Ollama handles CPU and GPU inference automatically, making local LLM "
            "deployment accessible without complex setup. Its API is compatible with the "
            "OpenAI Chat Completions specification."
        ),
        "source": "ollama_overview",
    },
]

print(f"📚 Knowledge base: {len(RAW_DOCUMENTS)} documents loaded.")

# ── Text chunking ─────────────────────────────────────────────────────────────
# For longer documents you would chunk them; our samples are already short.
# We show the splitter setup for completeness — it's a no-op on small texts.

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,          # characters per chunk
    chunk_overlap=60,        # overlap to preserve context across chunk boundaries
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = []
for doc in RAW_DOCUMENTS:
    for i, chunk_text in enumerate(splitter.split_text(doc["text"])):
        chunks.append({
            "id":     f"{doc['id']}_chunk{i}",
            "text":   chunk_text,
            "source": doc["source"],
        })

print(f"✂️  After chunking: {len(chunks)} text chunks ready for embedding.")
print(f"\n📝 Example chunk:\n   {chunks[0]['text'][:200]}...")

ModuleNotFoundError: No module named 'langchain.text_splitter'

---
## 🧮 Section 4: Embeddings and Vector Storage (ChromaDB)

**Embeddings** are dense numerical representations of text that capture semantic meaning. Texts with similar meanings have embeddings that are close together in vector space.

We use **`sentence-transformers/all-MiniLM-L6-v2`** — a small but effective embedding model:
- **Size:** ~22 MB (very lightweight)
- **Embedding dimension:** 384
- **Runs entirely on CPU** without any configuration

All embeddings are stored in a **ChromaDB** collection, which provides:
- Persistent local storage (no server needed)
- Fast approximate nearest-neighbour search
- Metadata filtering

In [ ]:
import chromadb
from chromadb.utils import embedding_functions
from tqdm import tqdm

# ── Embedding model ───────────────────────────────────────────────────────────
# Using SentenceTransformers via ChromaDB's built-in embedding function.
# The model is downloaded once and cached in ~/.cache/huggingface/

EMBEDDING_MODEL = "all-MiniLM-L6-v2"  # ~22 MB, excellent for CPU

print(f"Loading embedding model: {EMBEDDING_MODEL}")
print("(First run will download ~22 MB — subsequent runs use the cache)\n")

embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=EMBEDDING_MODEL
)

# ── ChromaDB setup ────────────────────────────────────────────────────────────
# PersistentClient stores the database on disk at ./chroma_db/
# Use chromadb.Client() for an in-memory-only version.

DB_PATH = "./chroma_db"  # Local directory for vector store persistence

chroma_client = chromadb.PersistentClient(path=DB_PATH)

# Create (or load existing) collection
# get_or_create_collection avoids errors if re-running the notebook
collection = chroma_client.get_or_create_collection(
    name="openvino_rag_demo",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"},  # Use cosine similarity
)

print(f"✅ ChromaDB collection ready at: {DB_PATH}")
print(f"   Collection name: 'openvino_rag_demo'")

In [ ]:
# ── Index documents ───────────────────────────────────────────────────────────
# Check how many documents are already in the collection to avoid duplicates.

existing_count = collection.count()

if existing_count >= len(chunks):
    print(f"ℹ️  Collection already contains {existing_count} documents. Skipping indexing.")
    print("   Delete './chroma_db/' and re-run to re-index.")
else:
    print(f"Indexing {len(chunks)} chunks into ChromaDB...")
    
    # Add all chunks in a single batch call for efficiency
    collection.add(
        ids       = [c["id"]     for c in chunks],
        documents = [c["text"]   for c in chunks],
        metadatas = [{"source": c["source"]} for c in chunks],
    )
    
    print(f"\n✅ Indexed {collection.count()} chunks successfully.")

print(f"\n📊 Vector store summary:")
print(f"   Total documents in collection: {collection.count()}")

---
## 🔍 Section 5: The Retrieval Step

Retrieval works by:

1. Converting the user's query into an embedding using the *same* embedding model used during indexing
2. Finding the `k` most similar document embeddings in ChromaDB using cosine similarity
3. Returning those documents as context

The key insight: **semantic similarity, not keyword matching**. A query about "fast inference" will retrieve documents about "optimized deployment" even if the exact words differ.

In [ ]:
def retrieve_documents(query: str, k: int = 3) -> list[dict]:
    """
    Retrieve the top-k most relevant document chunks for a given query.
    
    Args:
        query: The user's search question.
        k:     Number of documents to retrieve.
    
    Returns:
        List of dicts with keys: 'id', 'text', 'source', 'distance'
    """
    results = collection.query(
        query_texts=[query],
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )
    
    retrieved = []
    for i in range(len(results["ids"][0])):
        retrieved.append({
            "id":       results["ids"][0][i],
            "text":     results["documents"][0][i],
            "source":   results["metadatas"][0][i]["source"],
            "distance": results["distances"][0][i],  # Lower = more similar
        })
    
    return retrieved


# ── Test retrieval ────────────────────────────────────────────────────────────
test_query = "How does OpenVINO speed up AI inference?"
print(f"🔍 Test query: \"{test_query}\"\n")

retrieved = retrieve_documents(test_query, k=2)

for i, doc in enumerate(retrieved, 1):
    print(f"--- Result {i} (source: {doc['source']}, distance: {doc['distance']:.4f}) ---")
    print(f"{doc['text']}\n")

---
## ⚙️ Section 6: The RAG Pipeline

Now we connect retrieval with generation. The pattern is:

1. **Retrieve** relevant chunks for the user's question
2. **Format** a prompt that includes the retrieved context
3. **Generate** a response using the local LLM

A good system prompt instructs the LLM to:
- Answer only from the provided context
- Admit when it doesn't know (avoiding hallucination)
- Be concise

In [ ]:
def build_rag_prompt(query: str, context_docs: list[dict]) -> str:
    """
    Assemble the RAG prompt by combining retrieved context with the user query.
    
    Args:
        query:        The user's original question.
        context_docs: List of retrieved document dicts (from retrieve_documents).
    
    Returns:
        A formatted prompt string ready to send to the LLM.
    """
    context_str = "\n\n".join(
        f"[Source: {doc['source']}]\n{doc['text']}"
        for doc in context_docs
    )
    
    prompt = (
        f"You are a helpful assistant. Answer the question using ONLY the context "
        f"provided below. If the context does not contain enough information to answer, "
        f"say \"I don't have enough information to answer this question.\"\n\n"
        f"Context:\n{context_str}\n\n"
        f"Question: {query}\n\n"
        f"Answer:"
    )
    return prompt


def rag_query(query: str, k: int = 3) -> dict:
    """
    Full RAG pipeline: retrieve → build prompt → generate.
    
    Args:
        query: The user's question.
        k:     Number of documents to retrieve.
    
    Returns:
        Dict with 'query', 'retrieved_docs', and 'answer'.
    """
    # Step 1: Retrieve
    docs = retrieve_documents(query, k=k)
    
    # Step 2: Build prompt
    prompt = build_rag_prompt(query, docs)
    
    # Step 3: Generate
    answer = ask_llm(prompt)
    
    return {"query": query, "retrieved_docs": docs, "answer": answer}


# ── Run a test RAG query ───────────────────────────────────────────────────────
print("Running RAG pipeline...\n")

result = rag_query("What is ChromaDB and how is it used in AI applications?")

print(f"❓ Question: {result['query']}\n")
print(f"📄 Retrieved {len(result['retrieved_docs'])} documents:")
for doc in result['retrieved_docs']:
    print(f"   • {doc['source']} (similarity distance: {doc['distance']:.4f})")
print(f"\n💬 Answer:\n{result['answer']}")

---
## 🔄 Section 7: Minimal Agentic Loop with LangGraph

So far, our pipeline is a **static sequence**: always retrieve, always generate. An **agentic** pipeline adds reasoning and decision-making.

### How LangGraph Works

LangGraph models a workflow as a **state machine**:

- **State** — a typed dictionary that flows between nodes
- **Nodes** — Python functions that read/write the state
- **Edges** — connections between nodes (can be conditional)

### Our Agent Graph

```
START
  │
  ▼
[classify_query]   ← LLM decides: needs retrieval or direct answer?
  │         │
  │ (rag)   │ (direct)
  ▼         ▼
[retrieve]  [generate_direct]
  │                │
  ▼                │
[generate_rag]     │
  │                │
  └──────┬──────────┘
         ▼
       END
```

The agent first classifies the query: if it's a factual question that benefits from document lookup, it uses RAG; otherwise it answers directly. This avoids unnecessary retrieval for simple conversational or computational queries.

In [ ]:
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, START, END

# ── Agent State ───────────────────────────────────────────────────────────────
# Every node receives and returns a subset of this state dict.
# Using TypedDict gives us type hints and IDE support.

class AgentState(TypedDict):
    query:         str                  # Original user question
    route:         str                  # 'rag' or 'direct'
    retrieved_docs: list[dict]          # Documents from ChromaDB
    answer:        str                  # Final answer
    tool_result:   str                  # Optional: result of a tool call


print("✅ AgentState defined.")

In [ ]:
# ── Node 1: Query Classifier ──────────────────────────────────────────────────
# Asks the LLM to decide if this query needs document retrieval.

def classify_query(state: AgentState) -> AgentState:
    """
    Route the query:
    - 'rag'    → requires searching the knowledge base
    - 'direct' → can be answered without retrieval (math, simple facts, etc.)
    """
    decision_prompt = (
        f"You are a routing assistant. Given the user query below, decide if it "
        f"requires searching a knowledge base about AI tools (OpenVINO, ChromaDB, "
        f"Ollama, LangGraph, RAG) to answer correctly, or if it can be answered "
        f"directly.\n\n"
        f"Reply with ONLY one word: 'rag' or 'direct'.\n\n"
        f"Query: {state['query']}"
    )
    
    raw = ask_llm(decision_prompt).strip().lower()
    
    # Normalize the output — the LLM might add punctuation
    route = "rag" if "rag" in raw else "direct"
    
    print(f"   🗺️  Classifier decision: '{route}' (raw: '{raw}')")
    return {**state, "route": route}


# ── Node 2: Document Retrieval ────────────────────────────────────────────────

def retrieve_node(state: AgentState) -> AgentState:
    """Retrieve relevant documents from ChromaDB for the query."""
    docs = retrieve_documents(state["query"], k=3)
    print(f"   📚 Retrieved {len(docs)} documents.")
    return {**state, "retrieved_docs": docs}


# ── Node 3a: RAG Generation ───────────────────────────────────────────────────

def generate_rag_node(state: AgentState) -> AgentState:
    """Generate an answer grounded in the retrieved documents."""
    prompt   = build_rag_prompt(state["query"], state["retrieved_docs"])
    answer   = ask_llm(prompt)
    return {**state, "answer": answer}


# ── Node 3b: Direct Generation ────────────────────────────────────────────────

def generate_direct_node(state: AgentState) -> AgentState:
    """Generate an answer directly, without retrieval."""
    answer = ask_llm(
        prompt=state["query"],
        system="You are a helpful, concise assistant."
    )
    return {**state, "answer": answer, "retrieved_docs": []}


# ── Conditional edge: decides which generation node to call ───────────────────

def route_decision(state: AgentState) -> Literal["retrieve", "generate_direct"]:
    """Edge function: returns the name of the next node based on 'route'."""
    if state["route"] == "rag":
        return "retrieve"
    return "generate_direct"


print("✅ All graph nodes defined.")

In [ ]:
# ── Build the LangGraph state machine ─────────────────────────────────────────

builder = StateGraph(AgentState)

# Add nodes
builder.add_node("classify",         classify_query)
builder.add_node("retrieve",         retrieve_node)
builder.add_node("generate_rag",     generate_rag_node)
builder.add_node("generate_direct",  generate_direct_node)

# Entry point
builder.add_edge(START, "classify")

# Conditional routing after classification
builder.add_conditional_edges(
    "classify",
    route_decision,
    {
        "retrieve":        "retrieve",
        "generate_direct": "generate_direct",
    }
)

# After retrieval, always go to RAG generation
builder.add_edge("retrieve",        "generate_rag")

# Both generation nodes lead to END
builder.add_edge("generate_rag",    END)
builder.add_edge("generate_direct", END)

# Compile the graph into a runnable
agent = builder.compile()

print("✅ LangGraph agent compiled successfully.")
print("\nGraph structure:")
print("  START → classify → [retrieve → generate_rag | generate_direct] → END")

In [ ]:
def run_agent(query: str) -> str:
    """
    Run the agentic RAG pipeline for a given query.
    
    Args:
        query: Natural language question.
    
    Returns:
        The agent's final answer as a string.
    """
    print(f"\n{'='*60}")
    print(f"❓ Query: {query}")
    print(f"{'='*60}")
    
    # Initialize state with defaults
    initial_state: AgentState = {
        "query":          query,
        "route":          "",
        "retrieved_docs": [],
        "answer":         "",
        "tool_result":    "",
    }
    
    # Run the graph
    final_state = agent.invoke(initial_state)
    
    print(f"\n💬 Answer:\n{final_state['answer']}")
    
    if final_state["retrieved_docs"]:
        sources = list({d["source"] for d in final_state["retrieved_docs"]})
        print(f"\n📖 Sources consulted: {', '.join(sources)}")
    
    return final_state["answer"]


# ── Test 1: Knowledge-base question (should use RAG) ──────────────────────────
_ = run_agent("How does LangGraph help build AI agents?")

In [ ]:
# ── Test 2: Direct question (should skip retrieval) ───────────────────────────
_ = run_agent("What is 25 multiplied by 4?")

In [ ]:
# ── Test 3: Your own question ─────────────────────────────────────────────────
# Modify the query below to test with your own questions.

your_query = "What models does Ollama support and how does it compare to cloud APIs?"
_ = run_agent(your_query)

---
## 🛠️ Section 8: Optional — Adding a Simple Tool

One of the key features of an agentic pipeline is **tool use**: the ability to call external functions for tasks the LLM can't do reliably on its own (e.g., arithmetic, live search, code execution).

Here we add a minimal **calculator tool** as an example. The same pattern applies to any function: web search, database lookup, API calls, etc.

> **This section is self-contained and optional.** The core pipeline works without it.

In [ ]:
import re
import math

# ── Define a simple calculator tool ──────────────────────────────────────────

def calculator_tool(expression: str) -> str:
    """
    Safely evaluate a mathematical expression.
    Supports: +, -, *, /, **, sqrt(), sin(), cos(), pi, e
    
    Args:
        expression: A mathematical expression string.
    
    Returns:
        String representation of the result, or an error message.
    """
    # Whitelist safe names to prevent code injection
    safe_names = {
        "sqrt": math.sqrt, "sin": math.sin, "cos": math.cos,
        "tan": math.tan,   "log": math.log, "pi": math.pi,
        "e": math.e,       "abs": abs,      "round": round,
    }
    try:
        # Only allow digits, operators, parentheses, dots, and safe function names
        cleaned = re.sub(r"[^0-9+\-*/().^ a-zA-Z]", "", expression)
        result  = eval(cleaned, {"__builtins__": {}}, safe_names)  # noqa: S307
        return str(result)
    except Exception as exc:
        return f"Error: {exc}"


# ── Tool-augmented agent node ─────────────────────────────────────────────────

def tool_agent_query(query: str) -> str:
    """
    A simple tool-calling agent:
    1. Ask the LLM if this needs a calculator
    2. If yes, extract the expression and compute it
    3. Inject the result back into a final prompt
    """
    # Step 1: Detect if calculation is needed
    detection_prompt = (
        f"Does the following query require a mathematical calculation? "
        f"Reply with ONLY 'yes' or 'no'.\n\nQuery: {query}"
    )
    needs_calc = "yes" in ask_llm(detection_prompt).lower()
    
    tool_context = ""
    
    if needs_calc:
        # Step 2: Extract the math expression
        extract_prompt = (
            f"Extract ONLY the mathematical expression from this query as plain text. "
            f"Do not explain. Examples: '25 * 4', 'sqrt(144)', '2**10'\n\nQuery: {query}"
        )
        expression = ask_llm(extract_prompt).strip()
        calc_result = calculator_tool(expression)
        tool_context = f"Calculator result for '{expression}': {calc_result}\n\n"
        print(f"   🔧 Tool called: calculator('{expression}') → {calc_result}")
    
    # Step 3: Generate final answer using tool result if available
    final_prompt = (
        f"{tool_context}"
        f"Answer the following query concisely.\n\nQuery: {query}"
    )
    return ask_llm(final_prompt)


# ── Test the tool-augmented agent ─────────────────────────────────────────────
print("Testing tool-augmented agent:\n")

queries = [
    "What is the square root of 1764?",
    "If I have 256 tokens at 0.002 dollars each, what is the total cost?",
]

for q in queries:
    print(f"\n❓ {q}")
    answer = tool_agent_query(q)
    print(f"💬 {answer}")

---
## ⚡ Section 9: Optional — OpenVINO Integration

> **This section is informational and optional.** The notebook runs fully without OpenVINO installed. OpenVINO is an enhancement, not a requirement.

### Why Use OpenVINO for Local LLM Inference?

When running LLMs on Intel hardware (CPU, iGPU, NPU), OpenVINO can provide significant speedups through:

| Optimization | Description | Typical Benefit |
|---|---|---|
| **INT4 Quantization** | Reduce model weight precision 16-bit → 4-bit | 2–4× memory reduction |
| **INT8 Quantization** | Quantize activations during inference | 1.5–2× speedup |
| **KV-Cache Optimization** | Efficient attention cache memory layout | Faster long-context generation |
| **Graph Compilation** | Hardware-specific kernel fusion | Lower latency per token |

### Integration Approaches

**Option A: OpenVINO Model Server (OVMS)** — Drop-in replacement for Ollama/OpenAI API

```bash
# Convert and serve a Hugging Face model with INT4 quantization
pip install optimum[openvino]

optimum-cli export openvino \
    --model Qwen/Qwen2.5-3B-Instruct \
    --weight-format int4 \
    --output ./qwen2.5-3b-int4-ov
```

**Option B: `openvino-genai` Python API** — Direct inference without Ollama

```python
# pip install openvino-genai
import openvino_genai as ov_genai

pipe = ov_genai.LLMPipeline("./qwen2.5-3b-int4-ov", device="CPU")
result = pipe.generate("What is OpenVINO?", max_new_tokens=200)
print(result)
```

**Option C: LangChain + OpenVINO** — Plug into the existing RAG pipeline

```python
# pip install langchain-community openvino
from langchain_community.llms import HuggingFacePipeline
from optimum.intel import OVModelForCausalLM
from transformers import AutoTokenizer, pipeline

model_id = "./qwen2.5-3b-int4-ov"
tokenizer = AutoTokenizer.from_pretrained(model_id)
ov_model  = OVModelForCausalLM.from_pretrained(model_id)

ov_pipeline = pipeline("text-generation", model=ov_model, tokenizer=tokenizer)
llm = HuggingFacePipeline(pipeline=ov_pipeline)

# Drop-in replacement: use `llm` anywhere ask_llm() is called
```

### Hardware Support Matrix

| Hardware | OpenVINO Device | Notes |
|---|---|---|
| Intel CPU (Core / Xeon) | `CPU` | Fully supported, recommended for CPU-only |
| Intel iGPU (Iris Xe, Arc) | `GPU` | Requires OpenCL drivers |
| Intel NPU (Meteor Lake+) | `NPU` | Best for sustained generation tasks |
| ARM64 (e.g., Oracle A1) | `CPU` | Supported but no hardware-specific tuning |

### Checking OpenVINO Availability

In [ ]:
# ── Optional: Check if OpenVINO is available ──────────────────────────────────
# This cell gracefully handles the case where OpenVINO is not installed.

try:
    import openvino as ov
    
    core = ov.Core()
    available_devices = core.available_devices
    
    print("✅ OpenVINO is installed!")
    print(f"   Version: {ov.__version__}")
    print(f"   Available devices: {available_devices}")
    print()
    
    for device in available_devices:
        try:
            full_name = core.get_property(device, "FULL_DEVICE_NAME")
            print(f"   {device}: {full_name}")
        except Exception:
            print(f"   {device}: (details unavailable)")
    
    print()
    print("💡 To use OpenVINO for LLM inference, see the integration examples above.")
    print("   Recommended: openvino-genai with INT4-quantized Qwen2.5-3B")

except ImportError:
    print("ℹ️  OpenVINO is not installed — the pipeline above works without it.")
    print()
    print("   To install OpenVINO for accelerated Intel CPU/GPU inference:")
    print("   pip install openvino openvino-genai optimum[openvino]")
    print()
    print("   See: https://docs.openvino.ai/latest/get_started.html")

---
## 🎉 Section 10: Conclusion

Congratulations! You have built a complete **local Agentic RAG pipeline** from scratch. Here's what each component contributed:

| Component | Role | Key Benefit |
|---|---|---|
| **Ollama** | Local LLM inference server | Privacy, offline, no API costs |
| **ChromaDB** | Vector database | Fast semantic document retrieval |
| **`all-MiniLM-L6-v2`** | Embedding model | Lightweight, CPU-friendly |
| **LangGraph** | Agent orchestration | Flexible, stateful, loop-capable |
| **OpenVINO** *(optional)* | Inference optimization | Faster tokens on Intel hardware |

### 🧩 Pipeline Summary

```
User Query
    ↓
Classify: needs retrieval?
    ├── YES → ChromaDB similarity search → RAG prompt → Ollama LLM
    └── NO  → Direct prompt → Ollama LLM
    ↓
Answer (+ optional tool results)
```

### 🚀 Suggested Extensions

Here are practical ways to extend this notebook into a production-grade system:

1. **Load real documents** — Use `langchain.document_loaders` to ingest PDFs, web pages, or entire directories

2. **Add conversation memory** — Store chat history in LangGraph state to support multi-turn dialogue

3. **Upgrade the embedding model** — Try `BAAI/bge-small-en-v1.5` or `nomic-ai/nomic-embed-text-v1.5` for better retrieval quality

4. **Add more tools** — Web search (via DuckDuckGo API), code execution, calendar lookup

5. **Plug in OpenVINO** — Follow Section 9 to convert your Ollama model to OpenVINO IR format for faster CPU inference

6. **Add a Gradio UI** — Wrap the `run_agent()` function in a simple web interface with `gr.ChatInterface`

7. **Evaluate retrieval quality** — Use `ragas` library to measure faithfulness, answer relevancy, and context precision

### 📚 Further Reading

- [OpenVINO Documentation](https://docs.openvino.ai)
- [OpenVINO Notebooks Repository](https://github.com/openvinotoolkit/openvino_notebooks)
- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)
- [ChromaDB Documentation](https://docs.trychroma.com)
- [Ollama Model Library](https://ollama.com/library)
- [Qwen2.5 Model Card](https://huggingface.co/Qwen/Qwen2.5-3B-Instruct)

---
*This notebook was designed to follow [OpenVINO Notebooks](https://github.com/openvinotoolkit/openvino_notebooks) contribution standards: CPU-first, beginner-friendly, and fully reproducible on consumer hardware.*